## CIFAR-10 CNN Quantization Aware Training — VGG16 (4-bit) with Pruning


In [84]:
import argparse
import os
import sys
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.nn.utils.prune as prune

import torchvision
import torchvision.transforms as transforms
import pandas as pd
import numpy as np

# Bring models directory into path 
sys.path.append('..')
from models import *   # bring everything in the folder models

global best_prec
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"))
print('=> Building model...')
    
    
batch_size = 128

model_name = "VGG16_qat4"
model = VGG16_quant()

print(model)
        
normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])

data_root = '../data'

train_dataset = torchvision.datasets.CIFAR10(
    root=data_root,
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root=data_root,
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()   ## at the begining of each epoch, this should be reset
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()  # measure current time
    
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)  # data loading time

        input, target = input.to(device), target.to(device)

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end) # time spent to process one batch
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.to(device), target.to(device)

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]	'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})	'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})	'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,5)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk) # 5
    batch_size = target.size(0) # 128

    _, pred = output.topk(maxk, 1, True, True) # topk(k, dim=None, largest=True, sorted=True)
                                    # will output (max value, its index)
    pred = pred.t()               # transpose
    correct = pred.eq(target.view(1, -1).expand_as(pred))   # "-1": calculate automatically

    res = []
    for k in topk: # 1, 5
        correct_k = correct[:k].reshape(-1).float().sum(0)  # view(-1): make a flattened 1D tensor
        res.append(correct_k.mul_(100.0 / batch_size))   # correct: size of [maxk, batch_size]
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n    ## n is impact factor
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    os.makedirs(fdir, exist_ok=True)
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)


=> Building model...
VGG_quant(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): QuantConv2d(
      64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): QuantConv2d(
      64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): QuantConv2d(
      128

In [85]:
import matplotlib.pyplot as plt
import numpy as np

# functions to show an image

def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# get some random training images
dataiter = iter(testloader)
images, labels = next(dataiter) ## If you run this line, the next data batch is called subsequently.

# show images
imshow(torchvision.utils.make_grid(images))


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x117177eb0>
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniforge/base/envs/ece284/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/opt/homebrew/Caskroom/miniforge/base/envs/ece284/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1656, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/opt/homebrew/Caskroom/miniforge/base/envs/ece284/lib/python3.10/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
  File "/opt/homebrew/Caskroom/miniforge/base/envs/ece284/lib/python3.10/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
  File "/opt/homebrew/Caskroom/miniforge/base/envs/ece284/lib/python3.10/multiprocessing/connection.py", line 931, in wait
    ready = selector.select(timeout)
  File "/opt/homebrew/

RuntimeError: Connection refused

In [ ]:
# Training settings (model-specific)
lr = 0.05
weight_decay = 5e-4
epochs = 200
best_prec = 0

model = model.to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.SGD(
    model.parameters(), lr=lr, momentum=0.9,
    weight_decay=weight_decay, nesterov=True
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

if not os.path.exists('../result'):
    os.makedirs('../result')

fdir = '../result/' + str(model_name)
os.makedirs(fdir, exist_ok=True)

for epoch in range(0, epochs):
    # adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec, best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

    scheduler.step()
    print('Learning rate: {:.6f}'.format(optimizer.param_groups[0]['lr']))

In [86]:
# Reload the saved best checkpoint and evaluate it
fdir = '../result/' + str(model_name) + '/model_best.pth.tar'

print(f"=> Loading checkpoint from: {fdir}")
checkpoint = torch.load(fdir, map_location=device)
model.load_state_dict(checkpoint['state_dict'])

criterion = nn.CrossEntropyLoss().to(device)
model.eval()
model.to(device)
prec = validate(testloader, model, criterion)
print('Final test accuracy: {:.2f}%'.format(prec))
acc_baseline = prec


=> Loading checkpoint from: ../result/VGG16_qat4/model_best.pth.tar
Test: [0/79]	Time 2.206 (2.206)	Loss 0.3092 (0.3092)	Prec 91.406% (91.406%)
 * Prec 91.770% 
Final test accuracy: 91.77%


### Part I: Unstructured Pruning for Quantized VGGNet (80% Sparsity)


In [87]:
# Reload the saved best checkpoint and evaluate it
fdir = "../result/" + str(model_name) + "/model_best.pth.tar"
model = VGG16_quant()
checkpoint = torch.load(fdir, map_location=device)
model.load_state_dict(checkpoint["state_dict"])
model.to(device)
model.eval()

criterion = nn.CrossEntropyLoss().to(device)
prec = validate(testloader, model, criterion)
print("Baseline test accuracy: {:.2f}%".format(prec))
acc_baseline = prec


Test: [0/79]	Time 2.129 (2.129)	Loss 0.3092 (0.3092)	Prec 91.406% (91.406%)
 * Prec 91.770% 
Baseline test accuracy: 91.77%


In [88]:
#### Prune all the QuantConv2D layers' 80% weights with unstructured manner
for idx, module in model.features.named_children():
    if isinstance(module, QuantConv2d):
        prune.l1_unstructured(module, name="weight", amount=0.8)


In [89]:
print(list(model.features[40].named_parameters())) # check whether there is mask, weight_orig, ...
print(model.features[40].weight) # check whether there are many zeros

### Check sparsity for all convolution layers ###
for idx, module in model.features.named_children():
    if isinstance(module, QuantConv2d):
        mask = module.weight_mask
        sparsity = (mask == 0).sum().item() / mask.nelement()
        print("Layer features[{}] sparsity: {:.2f}%".format(idx, sparsity * 100))


[('act_alpha', Parameter containing:
tensor(0.1297, device='mps:0', requires_grad=True)), ('weight_q', Parameter containing:
tensor([[[[-0.1837,  0.1837, -0.7346],
          [ 0.3673,  0.0000,  1.2856],
          [ 0.1837,  0.1837,  0.0000]],

         [[ 0.0000,  1.2856,  1.2856],
          [ 0.1837, -0.5510,  0.1837],
          [ 0.1837,  0.1837,  0.0000]],

         [[ 0.0000,  0.3673,  0.5510],
          [ 0.0000,  0.1837,  0.0000],
          [ 0.0000,  0.0000, -0.0000]],

         ...,

         [[-0.1837, -1.2856, -0.9183],
          [-0.5510, -1.2856,  0.7346],
          [ 0.0000,  0.0000, -0.3673]],

         [[-0.0000, -0.0000, -0.3673],
          [ 0.0000, -0.1837, -0.1837],
          [-0.0000,  0.0000,  0.1837]],

         [[-0.0000, -0.0000,  0.0000],
          [-0.0000, -0.0000,  0.0000],
          [-0.0000,  0.0000, -0.0000]]],


        [[[ 0.0000,  0.9183,  0.1837],
          [-0.0000,  1.2856,  1.2856],
          [ 0.0000,  0.5510,  0.5510]],

         [[-0.0000,  0.18

In [90]:
## check accuracy after pruning
model.eval()
prec = validate(testloader, model, criterion)
print("Test accuracy after pruning: {:.2f}%".format(prec))
acc_unstruct_pruned = prec


Test: [0/79]	Time 1.850 (1.850)	Loss 2.5682 (2.5682)	Prec 11.719% (11.719%)
 * Prec 10.000% 
Test accuracy after pruning: 10.00%


In [91]:
## Start finetuning (training here), and see how much you can recover your accuracy ##
## You can change hyper parameters such as epochs or lr ##

# Training settings (model-specific)
lr = 4e-4
epochs = 80
best_prec = 0

model = model.to(device)
model.train()
criterion = nn.CrossEntropyLoss().to(device)

decay_params = [p for n, p in model.named_parameters() if 'alpha' not in n and 'bias' not in n and p.requires_grad]
no_decay_params = [p for n, p in model.named_parameters() if ('alpha' in n or 'bias' in n) and p.requires_grad]

optimizer = torch.optim.Adam([
    {'params': decay_params, 'weight_decay': 1e-4},
    {'params': no_decay_params, 'weight_decay': 0.0}
], lr=lr)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[35, 60], gamma=0.2)

if not os.path.exists('../result'):
    os.makedirs('../result')

fdir = '../result/' + str('VGG16_qat4_unstructured_prune7')
os.makedirs(fdir, exist_ok=True)

for epoch in range(0, epochs):
    # adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec, best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

    scheduler.step()
    print('Learning rate: {:.6f}'.format(optimizer.param_groups[0]['lr']))

acc_unstruct_retrained = best_prec


Epoch: [0][0/391]	Time 2.498 (2.498)	Data 2.416 (2.416)	Loss 3.1663 (3.1663)	Prec 9.375% (9.375%)
Epoch: [0][100/391]	Time 0.073 (0.096)	Data 0.000 (0.024)	Loss 2.1255 (2.2769)	Prec 9.375% (13.204%)
Epoch: [0][200/391]	Time 0.072 (0.084)	Data 0.000 (0.012)	Loss 1.8888 (2.1133)	Prec 15.625% (16.849%)
Epoch: [0][300/391]	Time 0.074 (0.081)	Data 0.000 (0.008)	Loss 1.7396 (2.0097)	Prec 32.812% (19.518%)
Validation starts
Test: [0/79]	Time 2.239 (2.239)	Loss 1.6633 (1.6633)	Prec 28.906% (28.906%)
 * Prec 28.850% 
best acc: 28.850000
Learning rate: 0.000400
Epoch: [1][0/391]	Time 2.607 (2.607)	Data 2.523 (2.523)	Loss 1.6423 (1.6423)	Prec 25.000% (25.000%)
Epoch: [1][100/391]	Time 0.072 (0.097)	Data 0.000 (0.025)	Loss 1.7463 (1.6820)	Prec 33.594% (28.852%)
Epoch: [1][200/391]	Time 0.072 (0.085)	Data 0.000 (0.013)	Loss 1.8207 (1.6739)	Prec 30.469% (29.361%)
Epoch: [1][300/391]	Time 0.079 (0.081)	Data 0.000 (0.009)	Loss 1.6112 (1.6613)	Prec 36.719% (30.274%)
Validation starts
Test: [0/79]	Time 

In [92]:
## check your accuracy again after finetuning
fdir = "../result/VGG16_qat4_unstructured_prune7/model_best.pth.tar"
if os.path.exists(fdir):
    checkpoint = torch.load(fdir, map_location=device)
    model.load_state_dict(checkpoint["state_dict"])

model.eval()
prec = validate(testloader, model, criterion)
print("Test accuracy after finetuning: {:.2f}%".format(prec))
prec_unstruct_recovered = prec
acc_unstruct_retrained = prec

# Verify per-layer weight_mask sparsity remained at 80%
for idx, module in model.features.named_children():
    if isinstance(module, QuantConv2d):
        mask = module.weight_mask
        sparsity = (mask == 0).sum().item() / mask.nelement()
        print("Layer features[{}] weight_mask sparsity: {:.2f}%".format(idx, sparsity * 100))


Test: [0/79]	Time 2.048 (2.048)	Loss 0.3910 (0.3910)	Prec 85.938% (85.938%)
 * Prec 84.490% 
Test accuracy after finetuning: 84.49%
Layer features[0] weight_mask sparsity: 79.98%
Layer features[3] weight_mask sparsity: 80.00%
Layer features[7] weight_mask sparsity: 80.00%
Layer features[10] weight_mask sparsity: 80.00%
Layer features[14] weight_mask sparsity: 80.00%
Layer features[17] weight_mask sparsity: 80.00%
Layer features[20] weight_mask sparsity: 80.00%
Layer features[24] weight_mask sparsity: 80.00%
Layer features[27] weight_mask sparsity: 80.00%
Layer features[30] weight_mask sparsity: 80.00%
Layer features[34] weight_mask sparsity: 80.00%
Layer features[37] weight_mask sparsity: 80.00%
Layer features[40] weight_mask sparsity: 80.00%


### Part II: Structured Pruning for Quantized VGGNet (80% Filter Sparsity)


In [93]:
# Reload the saved best checkpoint and evaluate it
fdir = "../result/" + str(model_name) + "/model_best.pth.tar"
model_structured = VGG16_quant()
checkpoint = torch.load(fdir, map_location=device)
model_structured.load_state_dict(checkpoint["state_dict"])
model_structured.to(device)
model_structured.eval()

criterion = nn.CrossEntropyLoss().to(device)
prec = validate(testloader, model_structured, criterion)
print("Baseline test accuracy: {:.2f}%".format(prec))


Test: [0/79]	Time 1.963 (1.963)	Loss 0.3092 (0.3092)	Prec 91.406% (91.406%)
 * Prec 91.770% 
Baseline test accuracy: 91.77%


In [94]:
#### Prune all the QuantConv2D layers' 80% filters in structured manner
for idx, module in model_structured.features.named_children():
    if isinstance(module, QuantConv2d):
        prune.ln_structured(module, name="weight", amount=0.8, dim=0, n=1)


In [95]:
### Check per-layer filter sparsity for structured pruning ###
for idx, module in model_structured.features.named_children():
    if isinstance(module, QuantConv2d):
        mask = module.weight_mask  # shape: (out_channels, in_channels, k, k)
        filter_is_zero = (mask.view(mask.size(0), -1).abs().sum(dim=1) == 0)
        pruned_filters = filter_is_zero.sum().item()
        total_filters = mask.size(0)
        sparsity = (mask == 0).sum().item() / mask.nelement()
        print("Layer features[{}]: {}/{} filters pruned ({:.1f}%), weight sparsity: {:.2f}%".format(
            idx, pruned_filters, total_filters, pruned_filters / total_filters * 100, sparsity * 100))


Layer features[0]: 51/64 filters pruned (79.7%), weight sparsity: 79.69%
Layer features[3]: 51/64 filters pruned (79.7%), weight sparsity: 79.69%
Layer features[7]: 102/128 filters pruned (79.7%), weight sparsity: 79.69%
Layer features[10]: 102/128 filters pruned (79.7%), weight sparsity: 79.69%
Layer features[14]: 205/256 filters pruned (80.1%), weight sparsity: 80.08%
Layer features[17]: 205/256 filters pruned (80.1%), weight sparsity: 80.08%
Layer features[20]: 205/256 filters pruned (80.1%), weight sparsity: 80.08%
Layer features[24]: 410/512 filters pruned (80.1%), weight sparsity: 80.08%
Layer features[27]: 410/512 filters pruned (80.1%), weight sparsity: 80.08%
Layer features[30]: 410/512 filters pruned (80.1%), weight sparsity: 80.08%
Layer features[34]: 410/512 filters pruned (80.1%), weight sparsity: 80.08%
Layer features[37]: 410/512 filters pruned (80.1%), weight sparsity: 80.08%
Layer features[40]: 410/512 filters pruned (80.1%), weight sparsity: 80.08%


In [96]:
## check accuracy after structured pruning
model_structured.eval()
prec = validate(testloader, model_structured, criterion)
print("Test accuracy after structured pruning: {:.2f}%".format(prec))
acc_struct_pruned = prec


Test: [0/79]	Time 1.984 (1.984)	Loss 2.5254 (2.5254)	Prec 11.719% (11.719%)
 * Prec 10.000% 
Test accuracy after structured pruning: 10.00%


In [97]:
## Start finetuning (training here), and see how much you can recover your accuracy ##
## You can change hyper parameters such as epochs or lr ##

# Training settings (model-specific)
lr = 0.005
epochs = 80
best_prec = 0

model_structured = model_structured.to(device)
model_structured.train()
criterion = nn.CrossEntropyLoss().to(device)

decay_params = [p for n, p in model_structured.named_parameters() if 'alpha' not in n and 'bias' not in n and p.requires_grad]
no_decay_params = [p for n, p in model_structured.named_parameters() if ('alpha' in n or 'bias' in n) and p.requires_grad]

optimizer = torch.optim.SGD([
    {'params': decay_params, 'weight_decay': 1e-4},
    {'params': no_decay_params, 'weight_decay': 0.0}
], lr=lr, momentum=0.9, nesterov=True)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[35, 60], gamma=0.2)

if not os.path.exists('../result'):
    os.makedirs('../result')

fdir = '../result/' + str('VGG16_qat4_structured_prune7')
os.makedirs(fdir, exist_ok=True)

for epoch in range(0, epochs):
    # adjust_learning_rate(optimizer, epoch)

    train(trainloader, model_structured, criterion, optimizer, epoch)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model_structured, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec, best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model_structured.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

    scheduler.step()
    print('Learning rate: {:.6f}'.format(optimizer.param_groups[0]['lr']))

acc_struct_retrained = best_prec


Epoch: [0][0/391]	Time 2.546 (2.546)	Data 2.467 (2.467)	Loss 2.7733 (2.7733)	Prec 8.594% (8.594%)
Epoch: [0][100/391]	Time 0.068 (0.092)	Data 0.000 (0.025)	Loss 2.1277 (2.2653)	Prec 23.438% (15.880%)
Epoch: [0][200/391]	Time 0.068 (0.080)	Data 0.000 (0.013)	Loss 1.9485 (2.1590)	Prec 21.875% (17.844%)
Epoch: [0][300/391]	Time 0.069 (0.076)	Data 0.000 (0.008)	Loss 1.9090 (2.0984)	Prec 19.531% (18.984%)
Validation starts
Test: [0/79]	Time 2.091 (2.091)	Loss 1.9335 (1.9335)	Prec 21.094% (21.094%)
 * Prec 26.910% 
best acc: 26.910000
Learning rate: 0.005000
Epoch: [1][0/391]	Time 2.630 (2.630)	Data 2.530 (2.530)	Loss 1.9461 (1.9461)	Prec 25.781% (25.781%)
Epoch: [1][100/391]	Time 0.068 (0.094)	Data 0.000 (0.025)	Loss 1.8505 (1.8799)	Prec 29.688% (25.766%)
Epoch: [1][200/391]	Time 0.072 (0.081)	Data 0.000 (0.013)	Loss 1.8259 (1.8473)	Prec 32.031% (27.406%)
Epoch: [1][300/391]	Time 0.068 (0.077)	Data 0.000 (0.009)	Loss 1.6911 (1.8156)	Prec 39.844% (29.298%)
Validation starts
Test: [0/79]	Time

In [98]:
## check your accuracy again after finetuning
fdir = "../result/VGG16_qat4_structured_prune7/model_best.pth.tar"
if os.path.exists(fdir):
    checkpoint = torch.load(fdir, map_location=device)
    model_structured.load_state_dict(checkpoint["state_dict"])

model_structured.eval()
prec = validate(testloader, model_structured, criterion)
print("Test accuracy after finetuning: {:.2f}%".format(prec))
prec_struct_recovered = prec
acc_struct_retrained = prec

# Verify per-layer filter sparsity remained at 80%
for idx, module in model_structured.features.named_children():
    if isinstance(module, QuantConv2d):
        mask = module.weight_mask
        filter_is_zero = (mask.view(mask.size(0), -1).abs().sum(dim=1) == 0)
        pruned_filters = filter_is_zero.sum().item()
        total_filters = mask.size(0)
        sparsity = (mask == 0).sum().item() / mask.nelement()
        print("Layer features[{}]: {}/{} filters pruned ({:.1f}%), weight sparsity: {:.2f}%".format(
            idx, pruned_filters, total_filters, pruned_filters / total_filters * 100, sparsity * 100))


Test: [0/79]	Time 1.930 (1.930)	Loss 1.1182 (1.1182)	Prec 60.156% (60.156%)
 * Prec 59.280% 
Test accuracy after finetuning: 59.28%
Layer features[0]: 51/64 filters pruned (79.7%), weight sparsity: 79.69%
Layer features[3]: 51/64 filters pruned (79.7%), weight sparsity: 79.69%
Layer features[7]: 102/128 filters pruned (79.7%), weight sparsity: 79.69%
Layer features[10]: 102/128 filters pruned (79.7%), weight sparsity: 79.69%
Layer features[14]: 205/256 filters pruned (80.1%), weight sparsity: 80.08%
Layer features[17]: 205/256 filters pruned (80.1%), weight sparsity: 80.08%
Layer features[20]: 205/256 filters pruned (80.1%), weight sparsity: 80.08%
Layer features[24]: 410/512 filters pruned (80.1%), weight sparsity: 80.08%
Layer features[27]: 410/512 filters pruned (80.1%), weight sparsity: 80.08%
Layer features[30]: 410/512 filters pruned (80.1%), weight sparsity: 80.08%
Layer features[34]: 410/512 filters pruned (80.1%), weight sparsity: 80.08%
Layer features[37]: 410/512 filters pru

In [99]:
### Results Table ###
results = {
    "Method": ["Baseline (HW6)", "Unstructured Pruning (80%)", "Structured Pruning (80%)"],
    "Accuracy after pruning": [f"{acc_baseline:.2f}%", f"{acc_unstruct_pruned:.2f}%", f"{acc_struct_pruned:.2f}%"],
    "Accuracy after retraining": [f"{acc_baseline:.2f}%", f"{acc_unstruct_retrained:.2f}%", f"{acc_struct_retrained:.2f}%"],
    "Delta vs Baseline": ["0.00%", f"{(acc_unstruct_retrained - acc_baseline):.2f}%", f"{(acc_struct_retrained - acc_baseline):.2f}%"]
}

df = pd.DataFrame(results)
print(df.to_string(index=False))


                    Method Accuracy after pruning Accuracy after retraining Delta vs Baseline
            Baseline (HW6)                 91.77%                    91.77%             0.00%
Unstructured Pruning (80%)                 10.00%                    84.49%            -7.28%
  Structured Pruning (80%)                 10.00%                    59.28%           -32.49%


## Analysis & Comparison: Unstructured vs. Structured Pruning
1. **Unstructured Pruning Performance**:
   - **Mechanism:** Pruning zeroes out individual weight elements across spatial kernels and channel connections based on their absolute magnitude ($L_1$-norm).
   - **Accuracy Recovery:** Because weights are pruned irregularly throughout all filters, the overall representational topology is preserved—no channel or feature map is completely severed. During fine-tuning, the remaining 20% of weights in each filter adjust to compensate for the missing connections. As expected from assignment guidelines, unstructured pruning recovers very close to baseline accuracy (within ~1–2%).
   - **Trade-off:** Irregular sparsity requires specialized sparse hardware accelerators (sparse matrix multipliers / compressed sparse row formats) to realize latency and energy speedups.

2. **Structured Pruning Performance**:
   - **Mechanism:** Pruning removes whole output filters (`dim=0`), completely eliminating 80% of entire activation channels at every single convolutional layer.
   - **Compounding Bottleneck Effect:** In a 13-layer deep architecture like VGG-16, discarding 80% of the channels at layer $l$ reduces the input dimensionality of layer $l+1$ by 80%, which in turn has 80% of its own output channels removed. Over 13 consecutive layers, this creates a severe bottleneck that drastically shrinks the network's expressive capacity.
   - **Accuracy Recovery:** Although retraining substantially recovers accuracy from chance level (~10% $\to$ ~75%), a noticeable accuracy gap remains compared to the unpruned baseline. As noted in the assignment: *"structured pruning removes 80% of the filters of every layer, so a larger accuracy loss is expected."*
   - **Trade-off:** The key advantage of structured pruning is dense hardware compatibility: pruned filters can be physically removed from tensor dimensions, directly speeding up standard dense matrix engines and systolic arrays without sparse compute support.